# Ling-3.0-tiny MLX QLoRA 微调

配套教程：`02-Ling-3.0-tiny MLX QLoRA 微调.md`

在 16GB 的 Apple Silicon Mac 上用 QLoRA 把 Ling-3.0-tiny 微调成「谁是卧底」桌游的
玩家决策组件：输入身份/词语/轮次历史，输出 ≤25 字发言或 `我投N号`/`我弃权` 投票。

## 1. 环境与模型准备

模型量化步骤同 01 notebook（4-bit 底座是 QLoRA 的前提）。

In [ ]:
%%bash
pip install "mlx-lm>=0.32.0"
# 若尚未量化：
# modelscope download --model inclusionAI/Ling-3.0-tiny --local_dir ~/models/Ling-3.0-tiny
# python -m mlx_lm.convert --model ~/models/Ling-3.0-tiny \
#     --q-bits 4 --mlx-path ~/models/Ling-3.0-tiny-4bit --trust-remote-code

## 2. 程序化合成指令数据

`mlx-lm` 训练数据是 chat 格式 JSONL（每行一个 `messages`）。下面是一个**可直接运行**
的迷你合成器：词对 × 模板生成对局样本，规则式打投票标签。生产级合成器（30 组词对、
属性池、历史窗口化）见完整教程。

In [ ]:
import json, random, os

rng = random.Random(42)

PAIRS = [  # (平民词, 卧底词)
    ("火锅", "麻辣烫"), ("可乐", "雪碧"), ("包子", "饺子"),
    ("牛奶", "豆浆"), ("咖啡", "奶茶"), ("自行车", "电动车"),
]
CIVILIAN_ATTRS = ["热气腾腾的", "有汤的", "冬天爱吃", "蘸料吃", "围坐一起吃"]
VAGUE_ATTRS = ["挺常见的", "很多人都喜欢", "味道不错", "到处都有"]

def synth_sample():
    civ_word, spy_word = rng.choice(PAIRS)
    is_spy = rng.random() < 0.2
    word, role = (spy_word, "卧底") if is_spy else (civ_word, "平民")
    task = rng.choice(["speak", "vote"])
    if task == "speak":
        if is_spy:
            reply = rng.choice(VAGUE_ATTRS) + "吧"
        else:
            reply = rng.choice(CIVILIAN_ATTRS) + "，你呢"
        user = (f"你在玩「谁是卧底」游戏。\n你的身份：{role}\n你的词语：{word}\n"
                f"规则：用一句话描述你的词语，不超过25个字，不要说出词语里的任何一个字。\n"
                f"请只输出描述内容：")
    else:
        vote = f"我投{rng.randint(1, 5)}号" if rng.random() < 0.6 else "我弃权"
        reply = vote
        user = (f"你在玩「谁是卧底」游戏。\n你的身份：{role}\n你的词语：{word}\n"
                f"规则：\n1. 根据大家的发言找出可疑的玩家\n"
                f"2. 只输出「我投N号」或「我弃权」，不要输出其他内容\n"
                f"历史发言：\n第1轮：\n1号：热气腾腾的\n2号：有汤的\n3号：挺常见的\n"
                f"现在开始投票。\n请只输出「我投N号」或「我弃权」：")
    return {"messages": [{"role": "user", "content": user},
                         {"role": "assistant", "content": reply}]}

os.makedirs("data", exist_ok=True)
for name, n in [("train", 800), ("valid", 80), ("test", 80)]:
    with open(f"data/{name}.jsonl", "w", encoding="utf-8") as f:
        for _ in range(n):
            f.write(json.dumps(synth_sample(), ensure_ascii=False) + "\n")
print("合成完成：data/train.jsonl 800 条 / valid 80 / test 80")
print(json.dumps(synth_sample(), ensure_ascii=False, indent=1)[:300])

## 3. QLoRA 训练

配置要点：4-bit 底座 + LoRA rank 8、只挂后 16/24 层（MoE 专家冻结，可训练 104M/1.3%）、
batch 1 × 梯度累积 4、`mask_prompt`（loss 只算输出）、梯度检查点。

In [ ]:
%%bash
cat > lora_config.yaml <<'EOF'
model: ~/models/Ling-3.0-tiny-4bit
train: true
data: ./data

fine_tune_type: lora
optimizer: adamw
num_layers: 16
batch_size: 1
grad_accumulation_steps: 4
iters: 1500
learning_rate: 1.0e-4
max_seq_length: 1024
mask_prompt: true
grad_checkpoint: true

adapter_path: ./adapters
save_every: 200
steps_per_report: 20
steps_per_eval: 100
val_batches: 16
seed: 42
trust_remote_code: true
EOF
python -m mlx_lm lora -c lora_config.yaml

> **16GB 实测**：连续训练约 625 步后确定性 `Metal Insufficient Memory`——**分段续训**，
> 每段 ≤600 步，用 `--resume-adapter-file ./adapters-seg1/0000600_adapters.safetensors`
> 接力（注意检查点是 **7 位**零填充）。推理只需 `adapters.safetensors` + `adapter_config.json`。

## 4. 推理验证

In [ ]:
%%bash
python -m mlx_lm.generate --model ~/models/Ling-3.0-tiny-4bit \
    --adapter-path ./adapters \
    --prompt "你在玩「谁是卧底」游戏。你的身份：平民。你的词语：火锅。用一句话描述你的词语，不超过25个字，不要说出词语里的任何一个字。" \
    --max-tokens 64

## 5. 常见问题

1. **输出带 `</thinking>` 前缀**：chat 模板渲染错位——推理用
   `apply_chat_template(..., thinking_option="off")`，解析剥模板残渣。权重没坏。
2. **贪心解码坍缩**（投票全弃权/发言全万能句）：用固定种子采样 temp=0.7, top_p=0.9。
3. **训练/推理分布对齐**：推理时折叠/截断输入的规则，训练数据必须覆盖其全部取值形态。
4. **量化 KV 勿开**（`--kv-bits 8`）：>2048 token 提示词会崩（ml-explore/mlx-lm#2057）。
